In [6]:
import math
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [7]:
# Lets use TDD rules methodology - this is full test, it will fail. Copy one line at the time until it passes.
x1 = Value(2.0, label='x1')
x2 = Value(0.0, label='x2')

w1 = Value(-3.0, label='w1')
w2 = Value(1.0, label='w2')

b = Value(6.8813735870195432, label='b')

x1w1 = x1*w1
x1w1.label = 'x1*w1'

x2w2 = x2*w2
x2w2.label = 'x2*w2'

x1w1x2w2 = x1w1 + x2w2
x1w1x2w2.label = 'x1w1 + x2w2'

n = x1w1x2w2 + b
n.label = 'n'

o = n.tanh() # I think I will stick to initial tanh activation function, makes sense to me
o.label = 'o'

o.backward()

NameError: name 'Value' is not defined

In [9]:
class Value:
    # lets implement Value class that I will use for micrograd
    def __init__(self, data, label, children=()):
        self.data = data
        self.label = label
        self._backward = lambda: None
        self.grad = 0
        self._children = set(children)

    def __mul__(self, other):
        result = Value(self.data * other.data, label = '*', children=(self, other))

        def _backward():
            self.grad += other.data * result.grad
            other.grad += self.data * result.grad

        result._backward = _backward
        return result

    def __add__(self, other):
        result = Value(self.data + other.data, label = '+', children=(self, other))

        def _backward():
            self.grad += result.grad
            other.grad += result.grad

        result._backward = _backward
        return result

    def tanh(self):
        result = Value(math.tanh(self.data), label = 'tanh', children=(self, ))

        def _backward():
            self.grad += (1 - result.data**2) * result.grad

        result._backward = _backward
        return result

    def backward(self):
        # 1. Lets construct full topological map of children, inputs should be first, n should be dead last
        topo = []
        visited = set()
        
        # 2. recursion here, not sure if python will like it
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                # children part
                for child in v._children: 
                    build_topo(child)
                topo.append(v)
        
        build_topo(self)
        
        # 3. Start the engine
        self.grad = 1.0 # I think this should be 1, Carpathy used grad as 1. Make sense for multiplication DC/DC = 1
        for node in reversed(topo):
            node._backward()


In [10]:
x1 = Value(2.0, label='x1')
x2 = Value(0.0, label='x2')

w1 = Value(-3.0, label='w1')
w2 = Value(1.0, label='w2')

b = Value(6.8813735870195432, label='b')

x1w1 = x1*w1
x1w1.label = 'x1*w1'

x2w2 = x2*w2
x2w2.label = 'x2*w2'

x1w1x2w2 = x1w1 + x2w2
x1w1x2w2.label = 'x1w1 + x2w2'

n = x1w1x2w2 + b
n.label = 'n'

o = n.tanh()
o.label = 'o'

o.backward()

In [11]:
print(f"x1 gradient: {x1.grad}")
print(f"x2 gradient: {x2.grad}")
print(f"w1 gradient: {w1.grad}")
print(f"w2 gradient: {w2.grad}")

x1 gradient: -1.4999999999999996
x2 gradient: 0.4999999999999999
w1 gradient: 0.9999999999999998
w2 gradient: 0.0
